# Stream Shaping

Before events reach a computation graph they often need cleaning: bad readings
filtered out, events below a threshold discarded, or a merged tagged stream
separated back into its constituent feeds.  screamer provides three
cardinality-changing operators: `dropna` removes rows whose value is NaN,
`filter` keeps only rows matching an arbitrary predicate, and `split`
partitions a source-tagged merged stream back into per-source streams.  All
three are fully causal — they make decisions based only on data observed so
far — and they compose naturally: `dropna(combine_latest(a, b))` is a common
idiom for warming up an aligned feed before feeding it downstream.

In [ ]:
import numpy as np
from screamer import dropna, filter as keep, split, merge

keys = np.array([1, 2, 3, 4, 5], dtype=np.int64)
vals = np.array([1.0, np.nan, 3.0, -4.0, 5.0])

## `dropna` — remove bad readings

`dropna` drops events whose value component contains NaN.  For scalar
streams (`how="any"` or `how="all"` are equivalent); for 2-D aligned
streams `how="any"` drops a row if *any* column is NaN (default), while
`how="all"` only drops it if *all* columns are NaN.  The output stream is
shorter — cardinality changes — but every surviving event has a clean value.

In [ ]:
# dropna: remove bad readings (cardinality shrinks)
ck, cv = dropna(keys, vals)

print("original keys :", keys)
print("original vals :", vals)
print()
print("clean keys    :", ck)   # key=2 (NaN) is gone
print("clean vals    :", cv)

assert not np.isnan(cv).any(), "no NaN should remain after dropna"
assert len(ck) == len(keys) - 1, "one NaN event was removed"

# Demonstrate how='all' on a 2-D aligned stream
# A row is kept if at least one column has a value
keys_2d = np.array([1, 2, 3], dtype=np.int64)
vals_2d = np.array([[1.0, np.nan],
                    [np.nan, np.nan],
                    [3.0, 4.0]])

k_any, v_any = dropna(keys_2d, vals_2d, how="any")  # drops rows 1 and 2
k_all, v_all = dropna(keys_2d, vals_2d, how="all")  # drops only row 2

print()
print("2-D how='any' keeps keys:", k_any, " (both columns must be non-NaN)")
print("2-D how='all' keeps keys:", k_all, " (drops only if ALL are NaN)")
assert list(k_any) == [3]
assert list(k_all) == [1, 3]

## `filter` — keep events matching a predicate

`filter` is the general escape hatch: it calls a Python predicate on each
event and keeps only those for which the predicate returns truthy.  The
row value is a scalar for 1-D streams and a 1-D NumPy array for 2-D streams.
Because the predicate is a Python callable it is flexible but not as fast as
a NumPy mask; prefer `dropna` for NaN removal and NumPy masking for
threshold cuts on large arrays.

In [ ]:
# filter: keep only events matching a predicate
# Here: keep positive values and reject NaN (NaN > 0 would be False anyway,
# but we guard explicitly to be safe with any future NaN policy)
pk, pv = keep(keys, vals, lambda v: (v > 0) if not np.isnan(v) else False)

print("original keys :", keys)
print("original vals :", vals)
print()
print("positive keys :", pk)   # key=2 (NaN) and key=4 (-4.0) removed
print("positive vals :", pv)

assert (pv > 0).all(), "only positive values should survive"
assert not np.isnan(pv).any(), "no NaN should survive"
assert len(pk) == 3, "keys 2 (NaN) and 4 (negative) were removed"

## `split` — inverse of `merge`

`split` takes the `(keys, values, sources)` triple that `merge` produces and
partitions it back into per-source `(keys, values)` pairs.  It is the exact
inverse of `merge`: `split(*merge(*series))` reconstructs the original
inputs.  The optional `n` parameter sets the number of output slots; pass it
explicitly to preserve empty sources.

In [ ]:
# split: inverse of merge — route a merged tagged stream back per source
a = (np.array([1, 3], dtype=np.int64), np.array([10., 30.]))
b = (np.array([2, 4], dtype=np.int64), np.array([20., 40.]))

mk, mv, ms = merge(a, b)
print("merged keys   :", mk)
print("merged values :", mv)
print("sources (0=a, 1=b):", ms)

parts = split(mk, mv, ms)

print()
print("split[0] keys :", parts[0][0], " values:", parts[0][1])
print("split[1] keys :", parts[1][0], " values:", parts[1][1])

# split inverts merge exactly
np.testing.assert_array_equal(parts[0][0], a[0])
np.testing.assert_array_equal(parts[0][1], a[1])
np.testing.assert_array_equal(parts[1][0], b[0])
np.testing.assert_array_equal(parts[1][1], b[1])
print("split(merge(a, b)) == (a, b): PASSED")

## Composing shaping operators

The operators compose naturally: `dropna(combine_latest(a, b))` is the
standard idiom for discarding the cold-start NaN rows that `emit="on_any"`
can produce.  The result is a clean, fully warm aligned stream ready for any
functor.

In [ ]:
from screamer import combine_latest

# Two async feeds
a_k = np.array([1, 3, 5], dtype=np.int64); a_v = np.array([10., 11., 12.])
b_k = np.array([2, 4, 6], dtype=np.int64); b_v = np.array([5., 6., 5.5])

# on_any starts immediately but has a NaN in the first row (feed b not yet seen)
k_raw, v_raw = combine_latest((a_k, a_v), (b_k, b_v), emit="on_any")
print("on_any (raw), first row has NaN for feed b:")
print("  keys  :", k_raw)
print("  values:", v_raw)

# dropna cleans the cold-start row, giving a fully warm aligned stream
k_clean, v_clean = dropna(k_raw, v_raw)
print()
print("after dropna (warm-up NaN removed):")
print("  keys  :", k_clean)
print("  values:", v_clean)

assert not np.isnan(v_clean).any()
print("clean & warm: PASSED")

**Takeaways**

- `dropna`, `filter`, and `split` all change cardinality — the output stream
  has fewer events than the input.  Keys are preserved so downstream operators
  always see correct logical timestamps.
- `dropna(how="any")` drops a row if *any* value component is NaN;
  `how="all"` only drops it when *all* components are NaN.  For 1-D streams
  the two modes are equivalent.
- `filter` accepts an arbitrary Python predicate; prefer `dropna` or NumPy
  masks for hot paths.
- `split` exactly inverts `merge`: `split(*merge(*series))` reconstructs the
  originals — useful for routing a mixed tagged stream back to per-symbol
  handlers.
- `dropna(combine_latest(..., emit="on_any"))` is the standard idiom for a
  warm aligned stream without any cold-start NaN rows.